# 06 · Reading the game's memory, and preparing it

**Goal:** understand how the repo turns the NES's raw 2 KB of RAM into things a fly could sense: *Mario's body*, *a list of objects with boxes*, and the *distances* between them.

You will learn
- where the facts live (`mario/facts/smb1.yaml`): RAM addresses, tile ids, object sizes. **Addresses are data, not code.**
- `read_mario` and the `ObjectInstance` / `Box` types from `contracts.py`
- the **object plug-ins** (`mario/objects/`): one file each, each with a `find(ram, facts)`
- how raw readings become *features* (the gap to the nearest enemy) — the input of the senses in notebook 03

No MaleCNS data needed. Notebook 05 is a good warm-up.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import _helpers as H
from fly_mario_fly import registry
from fly_mario_fly.contracts import Box
from fly_mario_fly.mario import ram as R
from fly_mario_fly.mario.env import MarioEnv

## 1. The facts table

`env.facts` loads `src/fly_mario_fly/mario/facts/smb1.yaml` (plus a per-level file). It holds every address and number about the game, with a comment
saying where it came from. Code elsewhere never hard-codes an address.

In [ ]:
env = MarioEnv()
step = env.reset("1-1", noop=3)
facts = env.facts

print("sections:", list(facts.keys()))
print()
for name, addr in list(facts["ram"].items())[:8]:
    print(f"{name:14s} 0x{addr:04X}")

## 2. Read bytes by hand

Mario's world x position is two bytes: a **page** (units of 256 pixels) and the x inside the page. Let's walk a bit and compute it ourselves,
then compare with the `x_pos` the gym wrapper reports.

In [ ]:
for _ in range(40):
    step = env.step(["right"])
ram = env.ram

page = int(ram[facts["ram"]["mario_page"]])
x_in_page = int(ram[facts["ram"]["mario_x"]])
print("page", page, "x in page", x_in_page, "->  world x =", page * 256 + x_in_page)
print("gym says       x_pos =", step.info["x_pos"])
print("repo helper    mario_x =", R.mario_x(ram, facts))

## 3. `read_mario`: Mario as a small object

`read_mario` packs a handful of bytes into a `MarioState` (see `contracts.py`): position, a box (width/height depend on his power state),
whether he's in the air, and the way he faces. **y is the top of his box; his feet are the reference** (ground is y = 176).

In [ ]:
mario = R.read_mario(ram, facts)
print(mario)
print("feet at y =", mario.y + mario.h, "(ground level is 176)")

## 4. Finding objects: the plug-ins

Every kind of thing the fly can be attached to is a **plug-in** in `src/fly_mario_fly/mario/objects/`: one file, one object called `PLUGIN`.
Each knows how to **find** itself in RAM and how to **remove** itself from the game (used for test scenes).
`registry.list_plugins("objects")` finds them all.

In [ ]:
objects = registry.list_plugins("objects")
for name, p in objects.items():
    print(f"{name:14s} kind={p.kind:9s} default loom shape={p.default_loom_shape:12s} {p.note[:60]}")

Let's walk until a Goomba is on the screen, then ask every plug-in what it can find. The answer is a list of
`ObjectInstance(type, id, box, side, state)`. A `Box` is in **level pixels**.

In [ ]:
def find_all(env):
    return {name: p.find(env.ram, env.facts) for name, p in objects.items()}

while not find_all(env)["goomba"]:
    step = env.step(["right"])

found = find_all(env)
for name, objs in found.items():
    for o in objs:
        print(f"{name:8s} id={o.id:<4} {o.side:7s} {o.box}")

`side` says where the box is relative to Mario: `ahead`, `behind`, `above`, `below`. The `state` dict carries extras
(a goomba's RAM slot and state, whether a Koopa is in its shell...).

Now draw them on the screen. Two offsets: the screen scrolls, so we subtract its left edge in the level (also stored in RAM), and level y starts **below the 32-px status bar**, so we add 32.

In [ ]:
STATUS_BAR = 32      # level y starts below the 32-px status bar (score, coins, time), so add it to draw on the screen

def scroll_x(ram, facts):
    r = facts["ram"]
    return int(ram[r["screen_page"]]) * 256 + int(ram[r["screen_x"]])

colours = {"goomba": "red", "pipe": "lime", "qblock": "yellow", "brick": "orange", "step": "cyan", "pit": "magenta", "koopa": "red"}

def draw_objects(env, frame, found, ax):
    sx = scroll_x(env.ram, env.facts)
    ax.imshow(frame, interpolation="nearest")
    for name, objs in found.items():
        for o in objs:
            b = o.box
            ax.add_patch(plt.Rectangle((b.x - sx - 0.5, b.y + STATUS_BAR - 0.5), b.w, b.h, fill=False, ec=colours.get(name, "white"), lw=1.8))
            ax.text(b.x - sx, b.y + STATUS_BAR - 2, name, color=colours.get(name, "white"), fontsize=7)
    ax.set_title(f"objects found in RAM (screen left edge = world x {sx})")
    ax.axis("off")

fig, ax = plt.subplots(figsize=(8, 7.5))
draw_objects(env, step.frame, found, ax)
plt.show()

Nothing here comes from the picture: all boxes come from RAM bytes plus the sizes in the facts file.

## 5. Terrain is a grid

Pipes, bricks and ?-blocks are not in the enemy slots; they are **tiles** in a 16 × 16 px grid that the game keeps in RAM (13 rows, scrolling columns).
`R.tile(ram, facts, world_x, row)` reads one cell. The ids (84 = ground, 81 = brick, ...) are listed in the facts file.

First walk to a screen with some terrain on it. We switch the Goombas off as we go (see section 7), so Mario survives until the first pipe.

In [ ]:
env.reset("1-1", noop=3)
while not objects["pipe"].find(env.ram, env.facts):
    env.step(["right"])
    objects["goomba"].remove(env)
step = env.step(["right"])
found = find_all(env)

fig, ax = plt.subplots(figsize=(8, 7.5))
draw_objects(env, step.frame, found, ax)
plt.show()
print({name: len(objs) for name, objs in found.items() if objs})

In [ ]:
cols = list(R.visible_columns(env.ram, facts))
grid = np.array([[R.tile(env.ram, facts, wx, row) for wx in cols] for row in range(13)])
print("tile grid:", grid.shape, "(rows x visible columns)")
print("ids that appear:", np.unique(grid))
print("facts say:", {k: v for k, v in facts["tiles"].items() if k in ("ground", "brick", "pipe", "qblock", "step")})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(step.frame[STATUS_BAR:], interpolation="nearest"); axes[0].set_title("screen (status bar cut off)"); axes[0].axis("off")
axes[1].imshow(grid, cmap="nipy_spectral", interpolation="nearest")
axes[1].set_title("tile ids in RAM (rows 0-12)")
plt.show()

The pipe plug-in turns the pipe tiles into **one box per pipe** with `R.merged_faces`: for each column it counts how tall the stack of pipe tiles is, and neighbouring columns of equal height are one pipe.

In [ ]:
print("merged faces (x0, x1, height in px):", R.merged_faces(env.ram, facts, facts["tiles"]["pipe"]))
print("is_solid(ground)?", R.is_solid(facts, facts["tiles"]["ground"]), "| is_solid(empty)?", R.is_solid(facts, 0))

## 6. Preparation: from boxes to a feature

A sense never looks at raw RAM. It gets `ObjectInstance`s and Mario, and computes what *a fly eye would see*. The first thing it needs is the **gap**
between Mario's box and the object's box. Let's record it frame by frame while Mario walks into a Goomba.

In [ ]:
def box_gap(a, b):
    """Smallest distance between two boxes (0 if they overlap): the same idea the looming sense uses."""
    gx = max(b.x - (a.x + a.w), a.x - (b.x + b.w), 0)
    gy = max(b.y - (a.y + a.h), a.y - (b.y + b.h), 0)
    return float(np.hypot(gx, gy))

env.reset("1-1", noop=3)
gaps, xs = [], []
while not env.done:
    env.step(["right"])
    m = R.read_mario(env.ram, env.facts)
    gob = objects["goomba"].find(env.ram, env.facts)
    if gob:
        mario_box = Box(m.x, m.y, m.w, m.h)
        gaps.append(min(box_gap(mario_box, g.box) for g in gob))
        xs.append(m.x)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(gaps)
ax.set_xlabel("frames since the goomba was first on screen"); ax.set_ylabel("gap to the goomba (px)")
ax.set_title("the number a looming sense starts from")
plt.show()
print("gap just before Mario died:", gaps[-1])

The gap shrinks steadily, then Mario dies. The senses turn this number (and the object's height) into an **angle** — what an eye would see growing. That is notebook 03.

## 7. Removing objects: test scenes

Each object plug-in can remove itself from the running game. The tool uses this to build *scenes* ("keep only the pits", "no enemies") so you can test one sense at a time.
Removing means writing a zero into the RAM slot (`env.poke`); the write is logged so a saved state replays it.

In [ ]:
env.reset("1-1", noop=3)
objects["goomba"].remove(env)                       # switch off the goombas that are currently active
steps = 0
while not env.done and steps < 600:
    env.step(["right"])
    objects["goomba"].remove(env)                   # (new ones scroll in as we go; remove them too)
    steps += 1
print("walked", steps, "frames; Mario is at x =", R.mario_x(env.ram, env.facts), "| done:", env.done)

Without Goombas the walk goes further, until something that is not a Goomba (here a pipe) stops him. Which `remove()` do the *terrain* plug-ins have? Open `mario/objects/pipe.py` and see.

## Recap

- RAM is just bytes; **`facts` says what the bytes mean**. Change a fact, not code.
- `read_mario` → `MarioState`; `plugin.find(ram, facts)` → `ObjectInstance(type, id, box, side, state)`.
- Enemies come from enemy slots; pipes/bricks/blocks come from the tile grid; both end up as the same `ObjectInstance`. That uniformity is what lets any sense attach to any object.
- A *feature* such as the gap is computed from those boxes — never from pixels.

**Try this**
1. Print `found["qblock"]` after walking further. What does the `state` of a brick contain?
2. Change `find_all` to also count `pit` objects. Walk to the first pit (around x ≈ 1380) and look at the `pit` box.
3. Open `mario/objects/koopa.py` and `mario/enemies.py`: what changes when a Koopa is in its shell?

**Next:** notebook 03 builds a *sense* from these readings. (Notebooks 01-02 and 04 look at the fly's brain.)